## 피처 60개 남기기

In [1]:
import pandas as pd

# ────────────────────────────────────────────
# 설정값 (여기만 수정하세요)
# ────────────────────────────────────────────
DATA_PATH    = r"C:\유비온프로젝트2\corporate-bankruptcy\혜원\M19_최종통합데이터(진).csv"      # 원본 데이터 경로
FEATURE_PATH = r"C:\유비온프로젝트2\corporate-bankruptcy\혜원\13번. 최종피처.parquet"          # 최종 피처 목록 경로
OUTPUT_PATH  = "M19_최종피처_60개.csv"           # 저장 경로
TARGET_COL   = "부실라벨_ICR3년"                 # 종속변수 (맨 끝에 유지)

# 제거할 컬럼 (기업 고유 식별자 등 불필요한 컬럼)
DROP_COLS = [
    '회사명',
    '사업자등록번호',
    '통계청 한국표준산업분류 코드 11차(대분류)',
    '통계청 한국표준산업분류 11차(중분류)',
    'M코드',
    '빅4감사',
]

# ────────────────────────────────────────────
# 1. 데이터 로드
# ────────────────────────────────────────────
df       = pd.read_csv(DATA_PATH)
features = pd.read_parquet(FEATURE_PATH)['final_feature'].tolist()

print(f"원본 데이터    : {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"선택된 피처 수 : {len(features)}개")

# ────────────────────────────────────────────
# 2. 누락 피처 확인
# ────────────────────────────────────────────
missing = [f for f in features if f not in df.columns]
if missing:
    print(f"\n⚠️  데이터에 없는 피처 ({len(missing)}개): {missing}")
else:
    print("✅ 60개 피처 모두 데이터에 존재")

# ────────────────────────────────────────────
# 3. 컬럼 선택
#    유지: 회계년도 + 60개 피처 + 종속변수
# ────────────────────────────────────────────
use_features = [f for f in features if f in df.columns]
keep_cols    = ['회계년도'] + use_features + [TARGET_COL]

df_filtered = df[keep_cols].copy()

print(f"\n필터링 후     : {df_filtered.shape[0]:,}행 × {df_filtered.shape[1]}열")
print(f"  └ 회계년도  : 1개")
print(f"  └ 피처      : {len(use_features)}개")
print(f"  └ 종속변수  : {TARGET_COL}")

# ────────────────────────────────────────────
# 4. 저장
# ────────────────────────────────────────────
df_filtered.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"\n저장 완료 → {OUTPUT_PATH}")

# ────────────────────────────────────────────
# 5. 타겟 분포 확인
# ────────────────────────────────────────────
dist = df_filtered[TARGET_COL].value_counts()
print(f"\n[타겟 분포]")
print(f"  정상(0) : {dist[0]:,}건 ({dist[0]/len(df_filtered)*100:.1f}%)")
print(f"  부실(1) : {dist[1]:,}건 ({dist[1]/len(df_filtered)*100:.1f}%)")

원본 데이터    : 39,932행 × 260열
선택된 피처 수 : 60개
✅ 60개 피처 모두 데이터에 존재

필터링 후     : 39,932행 × 62열
  └ 회계년도  : 1개
  └ 피처      : 60개
  └ 종속변수  : 부실라벨_ICR3년

저장 완료 → M19_최종피처_60개.csv

[타겟 분포]
  정상(0) : 38,433건 (96.2%)
  부실(1) : 1,499건 (3.8%)
